# Fuel Waste Prediction
### Original example by Danny Barren: *ML for Business Insights, Examples*

**The business question:** *On which trips are we burning more fuel than we should, and what is causing it?*

For a delivery or service fleet, fuel is often the second-largest cost after labour. Every trip has an **expected** fuel use (based on distance and vehicle) and an **actual** fuel use. The difference is **excess fuel**, our measure of waste.

This notebook trains a **HistGradientBoostingRegressor** (a fast, accurate tree-based model) to:
1. Predict excess fuel (litres) for each trip
2. Prove it beats a simple "use the average" baseline (MAE, RMSE, R²)
3. Translate the error into **dollars**
4. Rank the **drivers of waste** (idling, harsh braking, overdue maintenance...) so operations knows which levers to pull

**How to use it:** Edit only the **CONFIG** cell, then run top to bottom.
If `CSV_PATH = None`, a **synthetic** fleet dataset is generated so you can see every step working first.

In [ ]:
# ============================================================
# IMPORTS (scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import (train_test_split, KFold, RepeatedKFold,
                                     cross_validate, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: THE ONLY CELL YOU NEED TO EDIT
# ============================================================
CSV_PATH = None                     # e.g. "/content/fleet_trips.csv"; None = synthetic demo

ID_COLUMN = "trip_id"
TARGET_COLUMN = "excess_fuel_liters"  # actual fuel minus expected fuel for the trip (litres)

# Inputs known for each trip. Do NOT include the actual fuel used or anything
# computed from it (e.g. "fuel_cost", "mpg_actual"); that would be leakage.
NUMERIC_COLUMNS = ["distance_km", "idle_minutes", "avg_speed_kmh", "harsh_braking_events",
                   "payload_kg", "ambient_temp_c", "tire_pressure_dev_pct",
                   "vehicle_age_years", "driver_tenure_months", "days_since_service"]
CATEGORICAL_COLUMNS = ["vehicle_type", "route_type"]
DROP_COLUMNS = []

# --- Business numbers ----------------------------------------------
FUEL_PRICE_PER_LITER = 1.60         # your average fuel price
TRIPS_PER_YEAR = 50_000             # fleet-wide trips per year (for annual $ estimates)

MAIN_METRIC = "neg_mean_absolute_error"   # MAE: average size of a miss, in litres
DATA_SIZE_MODE = "auto"             # "auto", "small" (<1,000 rows), "large"
TEST_SIZE = 0.25
RANDOM_STATE = 42

In [ ]:
# ============================================================
# LOAD DATA (or build a synthetic fleet)
# ============================================================
def make_demo_trips(n=700, seed=RANDOM_STATE):
    """Synthetic trips. The story: idling, harsh braking, under-inflated tyres,
    overdue servicing and heavy loads waste fuel; experienced drivers waste less.
    Some effects are non-linear (waste jumps once service is > 90 days overdue)."""
    rng = np.random.default_rng(seed)
    vtype = rng.choice(["van", "box_truck", "pickup"], n, p=[0.5, 0.3, 0.2])
    route = rng.choice(["urban", "highway", "mixed"], n, p=[0.45, 0.25, 0.30])
    dist = np.round(rng.gamma(4, 30, n), 1)
    idle = np.round(rng.gamma(2, 9, n) * np.where(route == "urban", 1.6, 0.8), 1)
    speed = np.round(np.where(route == "highway", rng.normal(85, 8, n),
                     np.where(route == "urban", rng.normal(32, 6, n), rng.normal(55, 8, n))), 1)
    braking = rng.poisson(np.where(route == "urban", 6, 2))
    payload = np.round(rng.uniform(100, 2_500, n) * np.where(vtype == "box_truck", 1.5, 1.0))
    temp = np.round(rng.normal(15, 10, n), 1)
    tire = np.round(np.abs(rng.normal(0, 6, n)), 1)
    age = rng.integers(0, 12, n)
    tenure = rng.integers(1, 120, n)
    service = rng.integers(0, 200, n)
    excess = (0.12 * idle + 0.35 * braking + 0.18 * tire + 0.0008 * payload
              + 2.5 * (service > 90) + 0.01 * np.maximum(service - 90, 0)
              + 0.15 * age - 0.015 * np.minimum(tenure, 60)
              + 0.004 * dist * (vtype == "box_truck") + 0.03 * np.abs(temp - 18)
              + 0.02 * np.maximum(speed - 90, 0) ** 1.5 + rng.normal(0, 1.2, n))
    df = pd.DataFrame({"trip_id": np.arange(1, n + 1), "distance_km": dist, "idle_minutes": idle,
                       "avg_speed_kmh": speed, "harsh_braking_events": braking, "payload_kg": payload,
                       "ambient_temp_c": temp, "tire_pressure_dev_pct": tire, "vehicle_age_years": age,
                       "driver_tenure_months": tenure, "days_since_service": service,
                       "vehicle_type": vtype, "route_type": route,
                       "excess_fuel_liters": np.round(excess, 2)})
    for col in ["tire_pressure_dev_pct", "ambient_temp_c"]:   # sensor gaps
        df.loc[rng.random(n) < 0.04, col] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo_trips()
    print("Using SYNTHETIC demo data (CSV_PATH is None).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")
print(f"{len(df):,} trips x {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK
# ============================================================
needed = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
missing = [c for c in needed if c not in df.columns]
assert not missing, f"These CONFIG columns are not in your file: {missing}"

df = df.dropna(subset=[TARGET_COLUMN])            # we can't learn from trips without an answer
FEATURES = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X_all = df[FEATURES].copy()
for c in NUMERIC_COLUMNS:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce").astype(float)
for c in CATEGORICAL_COLUMNS:
    X_all[c] = X_all[c].astype("string").astype(object)
y_all = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
print(f"Average excess fuel per trip: {y_all.mean():.2f} L  (≈ ${y_all.mean() * FUEL_PRICE_PER_LITER:.2f})")
print("Config looks good.")

## 1. Light EDA: where does waste show up?

In [ ]:
display(pd.DataFrame({"missing_%": X_all.isna().mean() * 100, "unique": X_all.nunique()}).round(1))

fig, axes = plt.subplots(1, 1 + len(CATEGORICAL_COLUMNS), figsize=(5 * (1 + len(CATEGORICAL_COLUMNS)), 3.8))
axes[0].hist(y_all, bins=30, color="darkorange"); axes[0].set_title("Excess fuel per trip (L)")
for ax, col in zip(axes[1:], CATEGORICAL_COLUMNS):
    y_all.groupby(X_all[col]).mean().sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"Avg excess fuel by {col}")
plt.tight_layout(); plt.show()

# Quick correlation check: which numeric inputs move with waste? (a hint, not proof)
corr = X_all[NUMERIC_COLUMNS].corrwith(y_all).sort_values(key=abs, ascending=False)
display(corr.to_frame("correlation_with_waste"))
print("Leakage check: any correlation above 0.95 deserves a hard look:",
      corr[corr.abs() > 0.95].index.tolist() or "none found")

## 2. Split and validation plan
The test set is locked away until the end. With fewer than 1,000 trips we repeat 5-fold CV three times so the scores are steadier.

In [ ]:
size_mode = DATA_SIZE_MODE if DATA_SIZE_MODE != "auto" else ("small" if len(df) < 1_000 else "large")
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE)
cv = (RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE) if size_mode == "small"
      else KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
print(f"Data size mode: {size_mode} | train {len(X_train):,} trips, test {len(X_test):,} trips")

## 3. A leakage-safe pipeline for a tree model
Tree models don't need scaling. We only:
- fill missing numbers with the median (learned on training folds only)
- turn categories into integer codes and **tell the model they are categories** (so it can split on "urban vs. highway" directly)

Unseen categories in new data become `-1`, which the model treats as missing, so nothing crashes.

In [ ]:
preprocess = ColumnTransformer([
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("codes", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
     CATEGORICAL_COLUMNS),
    ("num", SimpleImputer(strategy="median"), NUMERIC_COLUMNS),
], verbose_feature_names_out=False)

# Categories come first in the output, so the first len(CATEGORICAL_COLUMNS) columns are categorical
is_categorical = [True] * len(CATEGORICAL_COLUMNS) + [False] * len(NUMERIC_COLUMNS)

hgb_pipe = Pipeline([("prep", preprocess),
                     ("model", HistGradientBoostingRegressor(categorical_features=is_categorical,
                                                             early_stopping=(size_mode == "large"),
                                                             random_state=RANDOM_STATE))])
hgb_pipe

## 4. Baseline vs. default model
**DummyRegressor** predicts the average waste for every trip. If our model can't beat that, it's not useful.
- **MAE**: average miss in litres (easy to explain)
- **RMSE**: like MAE but punishes big misses more
- **R²**: share of the trip-to-trip variation explained (0 = no better than average, 1 = perfect)

In [ ]:
scoring = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
def cv_summary(name, model):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
    return {"model": name, "MAE_L": -r["test_MAE"].mean(), "RMSE_L": -r["test_RMSE"].mean(),
            "R2": r["test_R2"].mean(), "train_R2_minus_valid_R2": r["train_R2"].mean() - r["test_R2"].mean()}

base = pd.DataFrame([cv_summary("Dummy (average waste)", DummyRegressor()),
                     cv_summary("HistGradientBoosting (defaults)", hgb_pipe)])
display(base)

## 5. Light tuning with RandomizedSearchCV
Instead of testing every combination, we try 25 random ones. The main dials:
- `learning_rate`: how big each correction step is (smaller = more careful)
- `max_leaf_nodes` / `max_depth`: how complex each tree can be
- `min_samples_leaf`: smallest group a rule may describe (bigger = steadier on small data)
- `l2_regularization`: a brake on extreme predictions

**Simplicity rule:** among settings within one standard error of the best MAE, keep the one with the **fewest leaves, then biggest leaf size**.

In [ ]:
param_dist = {
    "model__learning_rate": loguniform(0.02, 0.3),
    "model__max_iter": randint(100, 400),
    "model__max_leaf_nodes": randint(4, 32),
    "model__max_depth": [None, 3, 4, 6],
    "model__min_samples_leaf": randint(5, 40),
    "model__l2_regularization": loguniform(1e-3, 10),
}
search = RandomizedSearchCV(hgb_pipe, param_dist, n_iter=25, cv=cv, scoring=MAIN_METRIC,
                            random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)

res = pd.DataFrame(search.cv_results_)
best = res.loc[res["rank_test_score"].idxmin()]
se = best["std_test_score"] / np.sqrt(cv.get_n_splits())
close = res[res["mean_test_score"] >= best["mean_test_score"] - se].copy()
close = close.sort_values(["param_model__max_leaf_nodes", "param_model__min_samples_leaf"], ascending=[True, False])
chosen_params = close.iloc[0]["params"]

show = res[["mean_test_score", "std_test_score", "param_model__learning_rate", "param_model__max_leaf_nodes",
            "param_model__min_samples_leaf", "rank_test_score"]].sort_values("rank_test_score").head(6).copy()
show["mean_test_score"] = -show["mean_test_score"]
display(show.rename(columns={"mean_test_score": "CV_MAE_L"}))
print(f"Best CV MAE: {-best['mean_test_score']:.3f} L | chosen (simplest within 1 SE): {-close.iloc[0]['mean_test_score']:.3f} L")
print("Chosen settings:", {k.replace("model__", ""): (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in chosen_params.items()})

final_model = hgb_pipe.set_params(**chosen_params).fit(X_train, y_train)

## 6. The honest exam on the test set, in litres and dollars

In [ ]:
pred = final_model.predict(X_test)
dummy_pred = np.full(len(y_test), y_train.mean())
mae, rmse, r2 = mean_absolute_error(y_test, pred), np.sqrt(mean_squared_error(y_test, pred)), r2_score(y_test, pred)
mae_d = mean_absolute_error(y_test, dummy_pred)
print(f"Test MAE : {mae:.2f} L per trip (≈ ${mae * FUEL_PRICE_PER_LITER:.2f})  vs Dummy {mae_d:.2f} L")
print(f"Test RMSE: {rmse:.2f} L   |   Test R²: {r2:.2f}")
print(f"The model's typical miss is {(1 - mae / mae_d):.0%} smaller than just using the average.")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, pred, alpha=0.5)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, "k--", lw=1); ax.set_xlabel("actual excess fuel (L)"); ax.set_ylabel("predicted (L)")
ax.set_title("Predicted vs actual (closer to the line = better)"); plt.show()

## 7. What drives fuel waste?
**Permutation importance:** we shuffle one input at a time on the test set and measure how much worse the MAE gets. A big jump means the model relies on that input.
Then **partial dependence** plots show *how* waste changes as each top driver goes up, holding everything else at typical values.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="neg_mean_absolute_error",
                              n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"MAE_increase_L": perm.importances_mean, "spread": perm.importances_std},
                   index=FEATURES).sort_values("MAE_increase_L", ascending=False)
imp["MAE_increase_L"].iloc[::-1].plot.barh(figsize=(7, 4.5), color="teal")
plt.title("Permutation importance: increase in MAE when shuffled (L)"); plt.tight_layout(); plt.show()

top_drivers = [c for c in imp.index if c in NUMERIC_COLUMNS][:4]
fig, ax = plt.subplots(figsize=(14, 3.5))
PartialDependenceDisplay.from_estimator(final_model, X_train, top_drivers, ax=ax, n_cols=4)
plt.suptitle("How excess fuel changes as each driver increases", y=1.05); plt.tight_layout(); plt.show()

In [ ]:
# Plain-language summary of each top driver: compare predicted waste at low vs high values
print("In plain English:")
for col in top_drivers:
    lo, hi = X_train[col].quantile([0.1, 0.9])
    X_lo, X_hi = X_test.copy(), X_test.copy()
    X_lo[col], X_hi[col] = lo, hi
    diff = final_model.predict(X_hi).mean() - final_model.predict(X_lo).mean()
    per_year = diff * FUEL_PRICE_PER_LITER * TRIPS_PER_YEAR
    print(f"  • {col}: moving from a low value ({lo:,.1f}) to a high value ({hi:,.1f}) changes excess fuel by "
          f"{diff:+.2f} L per trip (≈ ${per_year:,.0f} per year if it applied to every trip).")

In [ ]:
# Action list: trips with the highest PREDICTED waste are where coaching / maintenance pays off first
scored = X_test.copy()
scored["predicted_excess_L"] = pred
scored["predicted_cost_$"] = pred * FUEL_PRICE_PER_LITER
if ID_COLUMN in df.columns:
    scored.insert(0, ID_COLUMN, df.loc[X_test.index, ID_COLUMN])
display(scored.sort_values("predicted_excess_L", ascending=False).head(10))
annual = y_all.clip(lower=0).mean() * FUEL_PRICE_PER_LITER * TRIPS_PER_YEAR
print(f"Estimated annual cost of excess fuel at current levels: ${annual:,.0f}")

## 8. What to do next: business recommendations
**Pull the levers the model points to** (in the demo: idling, overdue servicing, harsh braking, tyre pressure):
- **Idling:** set an engine-off policy and add idle alerts in telematics.
- **Maintenance:** enforce service intervals; the waste jump after the overdue line is usually the cheapest win.
- **Driving style:** target coaching at drivers whose trips repeatedly land in the top predicted-waste list.
- **Tyres:** add pressure checks to pre-trip inspections.

**Prove it:** pick one depot or team, apply one lever for 4–8 weeks, and compare actual excess fuel against a similar depot.

**Keep it healthy:** re-score weekly; retrain quarterly or when fuel blends, routes, or the vehicle mix change.

**Small vs. large data**
- *Small (< 1,000 trips):* repeated CV, larger `min_samples_leaf`, no early stopping. Read drivers as directional.
- *Large (100k+ trips):* early stopping is switched on automatically in `large` mode, a single 5-fold CV is enough, and you can sample rows for the search, then refit on everything.

> Correlation is not causation: the model shows which conditions *go with* waste. Confirm a lever with a small test before rolling it out fleet-wide.

In [ ]:
# Optional: save the pipeline for weekly scoring
import joblib
joblib.dump({"model": final_model, "features": FEATURES}, "fuel_waste_model.joblib")
print("Saved fuel_waste_model.joblib")
print(f"Summary: test MAE {mae:.2f} L vs {mae_d:.2f} L baseline, R² {r2:.2f}; top drivers: {', '.join(imp.index[:3])}.")